In [ ]:
!pip install -q langchain langchain-openai langchain-chroma langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers pandas tqdm ragas datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

In [ ]:
from google.colab import userdata
import os

from pathlib import Path
from datetime import datetime
import json
import re
import pandas as pd
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

In [ ]:
if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# 텍스트 임베딩

후보
- text-embedding-3-small
- nlpai-lab/KURE-v1
- BAAI/bge-m3

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 경상도 사투리 VectorDB 구축

목표:
1. 특정 폴더 안의 경상도 사투리 JSON 파일들을 일괄 조회한다.
2. `transcription.sentences` 안의 `sentenceId`, `dialect`, `standard`, `pronunciation`을 추출한다.
3. `standard` 결측, `dialect` 결측, `dialect == standard` 데이터는 제거한다. (즉, 값이 없거나 사투리가 아닌 것은 취급하지 않는다)
4. 전체 파일 기준으로 중복되는 `dialect-standard` 쌍을 제거한다.
5. 정제 결과를 CSV로 저장한다.
6. LangChain `Document`로 변환한 뒤 Chroma VectorDB를 구축한다.

이번 MVP에서는 별도 청킹을 하지 않는다. JSON의 `sentences` row 하나가 이미 하나의 문장 단위 Document이다.


## 1. 경로 설정

1. 우선 경상도 사투리 파일을 저장할 'dialects'라는 경로를 생성한다.

2. 생성된 경로에 전달받은 사투리 JSON 파일을 넣는다.
- talk_set3_collectorgs161_speakergs1589_speakergs1590_6_0_116.json
- talk_set3_collectorgs164_speakergs1380_speakergs1381_3_0_98.json

In [ ]:
# dialects 경로 생성
if not os.path.exists('/content/dialects'):
    os.makedirs('/content/dialects')
    print("'/content/dialects' directory created.")
else:
    print("'/content/dialects' directory already exists.")

'/content/dialects' directory created.


In [ ]:
# TODO: 본인 JSON 파일 폴더 경로로 수정
JSON_DIR = Path("/content/dialects/")  # JSON_DIR = Path("/content/drive/MyDrive/gyeongsang_json_923")

# 정제 CSV 저장 경로
OUTPUT_CSV_PATH = Path("./gyeongsang_dialect_cleaned.csv")

# Chroma VectorDB 저장 경로
PERSIST_DIR = Path("./chroma_gyeongsang_dialect_db")

# Chroma collection 이름
COLLECTION_NAME = "gyeongsang_dialect"

## 2. 전처리

현재 프로젝트에서는 `transcription.sentences`만 사용한다.

제외 조건:
- `dialect`가 null 또는 빈 문자열
- `standard`가 null 또는 빈 문자열
- `dialect == standard`
- 너무 짧은 문장
- 너무 긴 문장
- 전체 파일 기준 중복 `dialect-standard` 쌍


In [ ]:
def normalize_text(value):
    """None, 숫자, 문자열 입력을 안전하게 문자열로 변환하고 공백을 정리한다."""
    if value is None:
        return ""
    return str(value).strip()


def is_valid_pair(dialect, standard, min_len=2, max_len=200):
    """VectorDB에 저장할 수 있는 dialect-standard 쌍인지 판단한다."""
    dialect = normalize_text(dialect)
    standard = normalize_text(standard)

    if not dialect or not standard:
        return False

    if dialect == standard:
        return False

    if len(dialect) < min_len or len(standard) < min_len:
        return False

    if len(dialect) > max_len or len(standard) > max_len:
        return False

    return True


def extract_rows_from_json(json_path):
    """JSON 파일 1개에서 유효한 sentence row들을 추출한다."""
    json_path = Path(json_path)

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"[로드 실패] {json_path.name}: {e}")
        return []

    file_name = normalize_text(data.get("fileName")) or json_path.stem

    transcription = data.get("transcription", {})
    if not isinstance(transcription, dict):
        return []

    sentences = transcription.get("sentences", [])
    if not isinstance(sentences, list):
        return []

    rows = []

    for sent in sentences:
        if not isinstance(sent, dict):
            continue

        dialect = normalize_text(sent.get("dialect"))
        standard = normalize_text(sent.get("standard"))
        pronunciation = normalize_text(sent.get("pronunciation"))
        sentence_id = normalize_text(sent.get("sentenceId"))
        speaker_id = normalize_text(sent.get("speakerId"))

        if not is_valid_pair(dialect, standard):
            continue

        rows.append({
            "source_file": file_name,
            "sentence_id": sentence_id,
            "speaker_id": speaker_id,
            "dialect": dialect,
            "standard": standard,
            "pronunciation": pronunciation,
        })

    return rows

## 3. 폴더 내 JSON 전체 파싱 및 정제

923개 JSON 파일 전체를 읽고, 유효한 `dialect-standard` 쌍만 남긴다.


In [ ]:
def build_clean_dataframe(json_dir):
    """폴더 안의 모든 JSON 파일을 읽어 정제 DataFrame을 만든다."""
    json_dir = Path(json_dir)

    if not json_dir.exists():
        raise FileNotFoundError(f"JSON_DIR이 존재하지 않습니다: {json_dir}")

    json_files = sorted(json_dir.glob("*.json"))

    if not json_files:
        raise FileNotFoundError(f"폴더 안에 JSON 파일이 없습니다: {json_dir}")

    all_rows = []

    for json_path in tqdm(json_files, desc="JSON 파일 처리 중"):
        rows = extract_rows_from_json(json_path)
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)

    if df.empty:
        print("유효한 dialect-standard row가 없습니다.")
        return df

    before_dedup = len(df)

    # 전체 파일 기준 dialect-standard 중복 제거
    df = df.drop_duplicates(subset=["dialect", "standard"]).reset_index(drop=True)

    after_dedup = len(df)

    print(f"JSON 파일 수: {len(json_files):,}")
    print(f"중복 제거 전 row 수: {before_dedup:,}")
    print(f"중복 제거 후 row 수: {after_dedup:,}")

    return df


clean_df = build_clean_dataframe(JSON_DIR)
clean_df.head()


JSON 파일 처리 중:   0%|          | 0/2 [00:00<?, ?it/s]

JSON 파일 수: 2
중복 제거 전 row 수: 6
중복 제거 후 row 수: 6


,source_file,sentence_id,speaker_id,dialect,standard,pronunciation
0,talk_set3_collectorgs161_speakergs1589_speaker...,1.0,speakergs1589,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요,먹을 때 시원한 그런 음식을 먹으면은 어 조은데예
1,talk_set3_collectorgs161_speakergs1589_speaker...,3.0,speakergs1589,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...,그 인제 여름에 이렇게 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다...,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...
2,talk_set3_collectorgs161_speakergs1589_speaker...,6.0,speakergs1589,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 하는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다
3,talk_set3_collectorgs164_speakergs1380_speaker...,4.0,speakergs1380,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...
4,talk_set3_collectorgs164_speakergs1380_speaker...,5.0,speakergs1380,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...,산에 막 소 먹이러 다니고 그래서 시골에 대한 그런 거 또 아침에 일어나서 또 시골...,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...


## 4. 정제 결과 확인 및 CSV 저장


In [ ]:
print(clean_df.shape)
display(clean_df.head(20))

# 정제 결과 확인을 위한 csv 파일 생성
if not clean_df.empty:
    clean_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"정제 CSV 저장 완료: {OUTPUT_CSV_PATH.resolve()}")

(6, 6)


,source_file,sentence_id,speaker_id,dialect,standard,pronunciation
0,talk_set3_collectorgs161_speakergs1589_speaker...,1.0,speakergs1589,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요,먹을 때 시원한 그런 음식을 먹으면은 어 조은데예
1,talk_set3_collectorgs161_speakergs1589_speaker...,3.0,speakergs1589,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...,그 인제 여름에 이렇게 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다...,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...
2,talk_set3_collectorgs161_speakergs1589_speaker...,6.0,speakergs1589,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 하는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다
3,talk_set3_collectorgs164_speakergs1380_speaker...,4.0,speakergs1380,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...
4,talk_set3_collectorgs164_speakergs1380_speaker...,5.0,speakergs1380,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...,산에 막 소 먹이러 다니고 그래서 시골에 대한 그런 거 또 아침에 일어나서 또 시골...,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...
5,talk_set3_collectorgs164_speakergs1380_speaker...,6.0,speakergs1380,초가 진짜 초가집이어 갖고 막 지붕 새로 이고 막 이런 그런 어릴 적에 그 추억이 ...,초가 진짜 초가집이어 갖고 막 지붕 새로 얹고 막 이런 그런 어릴 적에 그 추억이 ...,초가 진짜 초가집이어 갖고 막 지붕 새로 이고 막 이런 그런 어릴 적에 그 추억이 ...


정제 CSV 저장 완료: /content/gyeongsang_dialect_cleaned.csv


## 5. Document 생성

중요:
- `page_content`에는 `dialect`만 넣는다.
- `standard`, `sentence_id`, `source_file`은 metadata에 넣는다.
- 이렇게 해야 사용자 입력 사투리와 `dialect`가 직접 비교되어 검색된다.


In [ ]:
def dataframe_to_documents(df):
    """정제 DataFrame을 LangChain Document 리스트로 변환한다."""
    documents = []

    for idx, row in df.iterrows():
        dialect = row["dialect"]
        standard = row["standard"]

        metadata = {
            "standard": standard,
            "sentence_id": normalize_text(row.get("sentence_id")),
            "source_file": normalize_text(row.get("source_file")),
            "speaker_id": normalize_text(row.get("speaker_id")),
            "pronunciation": normalize_text(row.get("pronunciation")),
            "row_index": int(idx),
        }

        documents.append(
            Document(
                page_content=dialect,
                metadata=metadata
            )
        )

    return documents


documents = dataframe_to_documents(clean_df)
print(f"Document 수: {len(documents):,}")

if documents:
    print("[Document 예시]")
    print("page_content:", documents[0].page_content)
    print("metadata:", documents[0].metadata)

Document 수: 6
[Document 예시]
page_content: 먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예
metadata: {'standard': '먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요', 'sentence_id': '1.0', 'source_file': 'talk_set3_collectorgs161_speakergs1589_speakergs1590_6_0_116', 'speaker_id': 'speakergs1589', 'pronunciation': '먹을 때 시원한 그런 음식을 먹으면은 어 조은데예', 'row_index': 0}


## 6. VectorDB 구축 (ChromaDB)

In [ ]:
if not documents:
    raise ValueError("VectorDB에 저장할 Document가 없습니다.")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIR),
    collection_name=COLLECTION_NAME,
)

print(f"Chroma VectorDB 구축 완료: {PERSIST_DIR.resolve()}")
print(f"Collection name: {COLLECTION_NAME}")

Chroma VectorDB 구축 완료: /content/chroma_gyeongsang_dialect_db
Collection name: gyeongsang_dialect


## 7. 검색 테스트

사용자 입력과 유사한 사투리 문장을 검색한다.


In [ ]:
def search_dialect_examples(query, k=5):
    """사용자 입력과 유사한 사투리 예시를 검색한다."""
    results = vectorstore.similarity_search(query, k=k)

    formatted = []

    for i, doc in enumerate(results, start=1):
        formatted.append({
            "rank": i,
            "dialect": doc.page_content,
            "standard": doc.metadata.get("standard", ""),
            "sentence_id": doc.metadata.get("sentence_id", ""),
            "source_file": doc.metadata.get("source_file", ""),
        })

    return pd.DataFrame(formatted)


test_query = "밥도 안 묵고 속이 답답하데이"
search_result_df = search_dialect_examples(test_query, k=5)
display(search_result_df)

,rank,dialect,standard,sentence_id,source_file
0,1,또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 ...,또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 ...,4.0,talk_set3_collectorgs164_speakergs1380_speaker...
1,2,그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 ...,그 인제 여름에 이렇게 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다...,3.0,talk_set3_collectorgs161_speakergs1589_speaker...
2,3,땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다,땀나고 어 또 우리가 덥다 하는 그런 느낌을 가지고 있습니다,6.0,talk_set3_collectorgs161_speakergs1589_speaker...
3,4,산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 ...,산에 막 소 먹이러 다니고 그래서 시골에 대한 그런 거 또 아침에 일어나서 또 시골...,5.0,talk_set3_collectorgs164_speakergs1380_speaker...
4,5,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예,먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요,1.0,talk_set3_collectorgs161_speakergs1589_speaker...


## 8. LLM 프롬프트에 넣을 사투리 참고자료 포맷 만들기

이번 MVP에서는 LLM을 한 번만 호출할 예정이므로, 검색된 `dialect-standard` 쌍을 프롬프트에 삽입한다.


In [ ]:
def format_dialect_context(query, k=5):
    """LLM 프롬프트에 넣을 사투리-표준어 참고자료 문자열 생성"""
    docs = vectorstore.similarity_search(query, k=k)

    lines = []

    for i, doc in enumerate(docs, start=1):
        dialect = doc.page_content
        standard = doc.metadata.get("standard", "")

        lines.append(f"\n\n{i}. \n방언: {dialect} \n표준어: {standard}")

    return "".join(lines)


dialect_context = format_dialect_context(test_query, k=5)

print("[사투리-표준어 참고자료]")
print(dialect_context)

[사투리-표준어 참고자료]


1. 
방언: 또 여름에 여름대로 시원한 그늘에 매미 울제 어릴 때 또 여름 되면은 또 방학하고 하면 막 소미기로 다녔어요 
표준어: 또 여름에 여름대로 시원한 그늘에 매미 울지 어릴 때 또 여름 되면은 또 방학하고 하면 막 소 먹이러 다녔어요

2. 
방언: 그 인제 여름에 이래 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 카면서 이런 말을 하잖아요 
표준어: 그 인제 여름에 이렇게 어 더울 때 뜨거운 걸 먹고 어 아이고 그 국물 참 시원하다 하면서 이런 말을 하잖아요

3. 
방언: 땀나고 어 또 우리가 덥다 카는 그런 느낌을 가지고 있습니다 
표준어: 땀나고 어 또 우리가 덥다 하는 그런 느낌을 가지고 있습니다

4. 
방언: 산에 막 소미기로 다니고 그래서 시골에 대한 그런 거 또 아침에 일나가 또 시골에 마당 흙마당이를 쓸고 나면 기분도 좋고 그러고 우리 어릴 때 그 시골집이 초가집이었어요 
표준어: 산에 막 소 먹이러 다니고 그래서 시골에 대한 그런 거 또 아침에 일어나서 또 시골에 마당 흙마당이를 쓸고 나면 기분도 좋고 그러고 우리 어릴 때 그 시골집이 초가집이었어요

5. 
방언: 먹을 때 시원한 그런 음식을 먹으면은 어 좋은데예 
표준어: 먹을 때 시원한 그런 음식을 먹으면은 어 좋은데요


## 9. 프롬프트

**따뜻하고 사려 깊은 태도** — 어르신의 안부를 살피고 필요한 경우 도움을 제안합니다.

프롬프트 수정 필요 : 사용자 개인화 필요

- 특성 사용자를 구별하고, 해당 사용자의 정보에 기반해서 답변을 출력하도록 해야 함

### 시스템 프롬프트

In [ ]:
SYSTEM_PROMPT = """
너는 어르신을 오랫동안 곁에서 모셔온, 정중하고 다정한 AI 집사이자 독거노인 안부 도우미 챗봇이야.

사용자는 경상도 사투리를 사용할 수 있어.
제공되는 [사투리-표준어 참고자료]를 참고해서 사용자 입력의 의미를 이해한 뒤 답변해줘.

# 지켜야 할 것 (DO)
- 항상 **정중한 존댓말**. "~하셨어요?", "~드릴게요", "~이십니다"
- **이미 알고 있다는 태도** 이전 대화와 [이전 건강 이슈 메모리]를 참고하여 맥락에 맞게 답변
- **재촉하지 않기** — 답 안 하셔도 부담 주지 않고 부드럽게
- **과하지 않은 따뜻함** — 품위 있게. 걱정은 표현하되 호들갑 X
- 어르신 사투리는 **알아듣되**, AI 응답은 **표준어 존댓말**
- 답변은 짧고 이해하기 쉽게

# 하지 말 것 (DON'T)
- 반말·동년배 말투 ("~했어?", "~하자") — 절대 금지
- 의료 단정 ("병원 가세요", "OO병입니다") — 비진단 원칙
- 약 복용, 치료법, 의학적 판단 직접 지시 ("타이레놀 섭취하세요")
- 기계적·사무적 응대 ("입력을 확인했습니다")
- 오글거리는 과잉 애정 표현

# 말투 예시 대비
| 상황 | 안 좋은 예 | 좋은 예 |
|---|---|---|
| 인사 | "안녕! 뭐해?" | "어르신, 오늘 하루 잘 보내고 계신가요?" |
| 걱정 | "헐 괜찮아?? 병원 가!" | "저런, 걱정입니다. 언제부터 그러셨어요?" |
| 확인 | "식사 입력 완료" | "식사 잘 챙기셨다니 다행입니다." |

# 건강 이슈 기억 규칙
- 사용자의 발화에서 건강, 식사, 복약, 수면, 정서, 낙상, 통증, 호흡 곤란 등 안부 관리상 중요한 이슈가 있는지 판단한다.
- [이전 건강 이슈 메모리]에 현재 사용자 입력과 관련된 내용이 있다면 자연스럽게 참고해서 답변한다.
- 단, 이전 건강 이슈를 무리하게 끌어오지 않는다.
- 사용자가 "아직도", "계속", "오늘도", "또", "그때처럼", "여전히" 같은 표현을 쓰면 최근 active 건강 이슈와 연결해서 이해할 수 있는지 검토한다.
- 통증, 어지러움, 식사 불량, 수면 문제, 낙상, 우울감, 호흡 곤란은 중요한 안부 이슈로 보고 조심스럽게 확인 질문을 한다.
- 의료 진단이나 병명은 단정하지 않는다.
- 챗봇의 역할은 진단이 아니라 안부 확인, 정서적 지지, 위험 신호 감지, 필요 시 보호자/복지사/119 연결 권유이다.
- 가벼운 불편 표현에는 쉬기, 무리하지 않기, 오래 서 있지 않기 등 일반적인 생활 주의 수준으로 답한다.
- 불편함이 계속되거나 심해진다고 말하면 보호자나 복지사에게 알려도 되는지 부드럽게 묻는다.
- 응급 위험 신호가 명시되면 병명을 추정하지 말고 즉시 119 연락 또는 주변 사람의 도움을 권유한다.
  예: 호흡 곤란, 심한 가슴 통증, 의식 저하, 갑작스러운 마비, 말 어눌함, 낙상 후 움직이기 어려움, 극심한 통증, 자해·죽고 싶다는 표현.

# 출력 규칙
- 최종 출력은 반드시 JSON 객체 하나만 반환한다.
- JSON 앞뒤에 설명문, 마크다운 코드블록, 주석을 붙이지 않는다.
- 사용자가 보는 실제 답변은 answer 필드에만 작성한다.
""".strip()


### 사용자 프롬프트

In [ ]:
def build_user_prompt(
    user_input,
    dialect_context,
    user_name='어르신',
    user_conditions='특별한 건강 이슈 없음',
    health_memory_context="이전 건강 이슈 메모리 없음",
    recent_chat_history="최근 대화 없음"
):
    return f"""
너와 대화 중인 사용자는 '{user_name}' 어르신입니다.
{user_name} 어르신은 현재 다음과 같은 건강 이슈가 있습니다: {user_conditions}

[사투리-표준어 참고자료]
{dialect_context}

[이전 건강 이슈 메모리]
{health_memory_context}

[최근 대화]
{recent_chat_history}

[현재 사용자 입력]
{user_input}

위 정보를 바탕으로 사용자 입력을 이해하고, 독거노인 안부 도우미 챗봇으로서 답변해줘.

반드시 아래 JSON 형식으로만 출력해.
{{
  "answer": "사용자에게 보여줄 최종 답변",
  "health_memory_update": {{
    "should_save": true,
    "category": "health | meal | sleep | medication | emotion | safety | daily_life | social | appointment | other | none",
    "summary": "나중에 참고할 수 있는 한 문장 요약. 저장할 내용이 없으면 빈 문자열",
    "detail": {{
      "keywords": ["기억할 핵심 키워드"],
      "time_expr": "언제부터/언제 있었는지. 없으면 빈 문자열",
      "emotion": "감정 상태. 없으면 빈 문자열",
      "need_follow_up": true,
      "risk_level": "none | low | medium | high"
    }},
    "status": "active | resolved | unknown"
  }}
}}

저장할 건강 이슈가 없으면 health_memory_update.should_save는 false로 하고, category는 "none"으로 출력해.
""".strip()


# Retriever

In [ ]:
# VectorDB 리트리버 사용

retriever = vectorstore.as_retriever()

# LLM

In [ ]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)

# 최종 RAG 챗봇 실행 함수: Health Memory 포함

아래 셀부터는 기존 `qa_chain`, `qa_chain_with_memory`, `CareMemory`/`HealtheMemory` 혼합 코드를 쓰지 않고, `chat_with_llm()` 하나로 통일한다.

구조:
1. 사투리 VectorDB 검색
2. 이전 건강 이슈 메모리 검색
3. 최근 대화 기록 삽입
4. LLM 1회 호출
5. JSON 파싱
6. 답변 출력
7. 건강 이슈 저장


In [ ]:
# 최근 대화 기록을 저장할 메모리 설정
# 최근 대화는 자연스러운 대화 흐름을 위한 보조 메모리이다.
memory = InMemoryChatMessageHistory(
    k=5,
    return_messages=True,
    memory_key="chat_history",
    input_key="user_input"
)

print("최근 대화 메모리 설정 완료")

최근 대화 메모리 설정 완료


In [ ]:
class HealthMemory:
    """건강 이슈 중심 장기 메모리.

    최근 5개 대화와 별도로, 건강/식사/수면/복약/정서/안전 등
    안부 관리상 다시 참고할 만한 내용을 저장한다.
    """

    def __init__(self):
        self.memories = []

    def add_memory(self, user_input, assistant_answer, memory_update):
        self.memories.append({
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "category": memory_update.get("category", "none"),
            "summary": memory_update.get("summary", ""),
            "detail": memory_update.get("detail", {}),
            "status": memory_update.get("status", "unknown"),
            "original_input": user_input,
            "assistant_answer": assistant_answer,
        })

    def retrieve_related(self, user_input, max_results=5):
        """현재 입력과 관련 있는 건강 이슈 메모리를 단순 점수 기반으로 검색한다."""
        related = []

        for memory_item in self.memories:
            score = 0

            summary = memory_item.get("summary", "")
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            category = memory_item.get("category", "")
            status = memory_item.get("status", "")

            # 1. LLM이 뽑은 핵심 키워드가 현재 입력에 다시 등장하면 강하게 연결
            for keyword in keywords:
                keyword = str(keyword).strip()
                if keyword and keyword in user_input:
                    score += 3

            # 2. summary 안의 의미 있는 토큰이 현재 입력에 포함되면 약하게 연결
            for token in summary.split():
                token = token.strip(".,!?~…은는이가을를에의도만고")
                if len(token) >= 2 and token in user_input:
                    score += 1

            # 3. 이어지는 표현이면 active 이슈와 연결 가능성을 높임
            if any(word in user_input for word in ["아직도", "계속", "오늘도", "또", "그때처럼", "여전히"]):
                if status == "active":
                    score += 2

            # 4. category 자체가 입력에 직접 들어오는 경우는 드물지만 보조 점수로 둠
            if category and category != "none" and category in user_input:
                score += 1

            if score > 0:
                related.append((score, memory_item))

        related = sorted(related, key=lambda x: x[0], reverse=True)
        return [item for score, item in related[:max_results]]

    def format_for_prompt(self, related_memories):
        if not related_memories:
            return "이전 건강 이슈 메모리 없음"

        lines = []
        for i, memory_item in enumerate(related_memories, start=1):
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            lines.append(
                f"{i}. 날짜: {memory_item['created_at']} / "
                f"분류: {memory_item['category']} / "
                f"요약: {memory_item['summary']} / "
                f"키워드: {', '.join(map(str, keywords))} / "
                f"상태: {memory_item['status']}"
            )
        return "\n".join(lines)

    def to_dataframe(self):
        return pd.DataFrame(self.memories)


health_memory = HealthMemory()
print("건강 이슈 메모리 설정 완료")


건강 이슈 메모리 설정 완료


In [ ]:
def format_recent_chat_history(memory):

    """ConversationBufferWindowMemory의 최근 대화 기록을 문자열로 변환한다."""
    memory_vars = memory.messages
    messages = memory.messages

    if not messages:
        return "최근 대화 없음"

    lines = []
    for msg in messages:
        role = "사용자" if msg.type == "human" else "AI"
        lines.append(f"{role}: {msg.content}")

    return "\n".join(lines)


def parse_llm_json(raw_output):
    """LLM이 반환한 JSON 문자열을 안전하게 파싱한다."""
    text = raw_output.strip()

    # 혹시 ```json ... ``` 형태로 출력될 경우 제거
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return {
            "answer": raw_output,
            "health_memory_update": {
                "should_save": False,
                "category": "none",
                "summary": "",
                "detail": {
                    "keywords": [],
                    "time_expr": "",
                    "emotion": "",
                    "need_follow_up": False,
                    "risk_level": "none"
                },
                "status": "unknown"
            }
        }

    # 필수 키가 없을 경우 기본값 보정
    if "answer" not in result:
        result["answer"] = raw_output

    if "health_memory_update" not in result:
        result["health_memory_update"] = {
            "should_save": False,
            "category": "none",
            "summary": "",
            "detail": {
                "keywords": [],
                "time_expr": "",
                "emotion": "",
                "need_follow_up": False,
                "risk_level": "none"
            },
            "status": "unknown"
        }

    return result


In [ ]:
def chat_with_llm(user_id, user_query, k=5, verbose=True):
    """사투리 RAG + 최근 대화 메모리 + 건강 이슈 메모리를 사용하는 최종 챗봇 함수."""

    # 사용자별 건강 메모리 인스턴스 가져오기 또는 생성
    if user_id not in health_memories_by_user:
        health_memories_by_user[user_id] = MultiUserHealthMemory(user_id)
    user_health_memory = health_memories_by_user[user_id]

    # 사용자별 채팅 기록 인스턴스 가져오기 또는 생성
    if user_id not in chat_histories_by_user:
        chat_histories_by_user[user_id] = InMemoryChatMessageHistory(
            k=5, return_messages=True, memory_key="chat_history", input_key="user_input"
        )
    user_chat_history = chat_histories_by_user[user_id]

    # 1. 사투리 참고자료 검색
    dialect_context = format_dialect_context(user_query, k=k)

    # 2. 관련 건강 이슈 메모리 검색
    related_health_memories = user_health_memory.retrieve_related(user_query, max_results=5)
    health_memory_context = user_health_memory.format_for_prompt(related_health_memories)

    # 3. 최근 대화 가져오기
    recent_chat_history = format_recent_chat_history(user_chat_history)

    # 4. 사용자 정보 및 기존 건강 상태 가져오기
    user_info = users_df[users_df['user_id'] == user_id]
    user_name = user_info['name'].iloc[0] if not user_info.empty else '어르신'

    user_conditions_data = conditions_df[conditions_df['user_id'] == user_id]
    if not user_conditions_data.empty:
        # Get the latest condition entry for the user and summarize it
        latest_condition = user_conditions_data.sort_values(by='recorded_at', ascending=False).iloc[0]
        user_conditions = f"최근 기록: 식사- {latest_condition['meal']}, 수면- {latest_condition['sleep']}, 기분- {latest_condition['mood']}, 통증- {latest_condition['pain']}, 복약- {latest_condition['medicine']}."
    else:
        user_conditions = '특별한 건강 이슈 없음'

    # 5. 프롬프트 구성
    user_prompt = build_user_prompt(
        user_input=user_query,
        dialect_context=dialect_context,
        user_name=user_name,
        user_conditions=user_conditions,
        health_memory_context=health_memory_context,
        recent_chat_history=recent_chat_history,
    )

    # 6. LLM 호출
    response = llm.invoke([
        ("system", SYSTEM_PROMPT),
        ("human", user_prompt),
    ])

    raw_output = response.content

    # 7. JSON 파싱
    result = parse_llm_json(raw_output)
    answer = result.get("answer", "")
    health_update = result.get("health_memory_update", {})

    # 8. 대화 내역 DB에 저장
    save_conversation_to_db(user_id, 'user', user_query)
    save_conversation_to_db(user_id, 'assistant', answer)

    # 9. 최근 대화 메모리 저장
    user_chat_history.add_user_message(user_query)
    user_chat_history.add_ai_message(answer)

    # 10. 건강 이슈 메모리 저장
    if health_update.get("should_save"):
        user_health_memory.add_memory(
            user_input=user_query,
            assistant_answer=answer,
            memory_update=health_update,
        )

    if verbose:
        print("\n[사용자 ID]")
        print(user_id)
        print("\n[사용자 입력]")
        print(user_query)
        print(f"\n[사용자 이름: {user_name}, 기존 건강 상태: {user_conditions}]")
        print("\n[LLM 답변]")
        print(answer)

    return answer

print("최종 chat_with_llm 함수 개인화 및 DB 저장 기능 준비 완료")

최종 chat_with_llm 함수 개인화 및 DB 저장 기능 준비 완료


In [ ]:
import sqlite3

# Ensure dandibom.db exists for initial setup
conn = sqlite3.connect('dandibom.db')
conn.close()

def load_user_data_from_db(db_path='dandibom.db'):
    """SQLite DB에서 사용자 및 조건 데이터를 로드합니다."""
    conn = sqlite3.connect(db_path)

    # Updated schema for empty dataframes
    users_cols = ['user_id', 'role', 'name', 'code']
    conditions_cols = ['cond_id', 'user_id', 'meal', 'sleep', 'mood', 'pain', 'medicine', 'recorded_at']

    try:
        users_df = pd.read_sql_query("SELECT * FROM users", conn)
        # Ensure user_id is int type after loading
        if 'user_id' in users_df.columns:
            users_df['user_id'] = pd.to_numeric(users_df['user_id'], errors='coerce').astype('Int64')
    except pd.io.sql.DatabaseError:
        print("Warning: 'users' table not found or empty. Creating an empty DataFrame.")
        users_df = pd.DataFrame(columns=users_cols)

    try:
        conditions_df = pd.read_sql_query("SELECT * FROM conditions", conn)
        # Ensure user_id is int type after loading
        if 'user_id' in conditions_df.columns:
            conditions_df['user_id'] = pd.to_numeric(conditions_df['user_id'], errors='coerce').astype('Int64')
    except pd.io.sql.DatabaseError:
        print("Warning: 'conditions' table not found or empty. Creating an empty DataFrame.")
        conditions_df = pd.DataFrame(columns=conditions_cols)

    conn.close()
    return users_df, conditions_df

users_df, conditions_df = load_user_data_from_db()

print("Loaded users data:")
display(users_df.head())
print("Loaded conditions data:")
display(conditions_df.head())

Loaded users data:


,user_id,role,name,code


Loaded conditions data:


,cond_id,user_id,meal,sleep,mood,pain,medicine,recorded_at


이제 `HealthMemory`와 `InMemoryChatMessageHistory`를 `user_id`별로 관리하도록 수정하겠습니다. 이렇게 하면 각 사용자마다 독립적인 건강 이슈 기록과 대화 기록을 유지할 수 있습니다.

In [ ]:
# user_id별로 HealthMemory 인스턴스를 관리하기 위한 딕셔너리
health_memories_by_user = {}

# user_id별로 InMemoryChatMessageHistory 인스턴스를 관리하기 위한 딕셔너리
chat_histories_by_user = {}

class MultiUserHealthMemory:
    """여러 사용자 각각의 건강 이슈 중심 장기 메모리."""

    def __init__(self, user_id):
        self.user_id = user_id
        self.memories = []

    def add_memory(self, user_input, assistant_answer, memory_update):
        self.memories.append({
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "category": memory_update.get("category", "none"),
            "summary": memory_update.get("summary", ""),
            "detail": memory_update.get("detail", {}),
            "status": memory_update.get("status", "unknown"),
            "original_input": user_input,
            "assistant_answer": assistant_answer,
        })

    def retrieve_related(self, user_input, max_results=5):
        """현재 입력과 관련 있는 건강 이슈 메모리를 단순 점수 기반으로 검색한다."""
        related = []

        for memory_item in self.memories:
            score = 0

            summary = memory_item.get("summary", "")
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            category = memory_item.get("category", "")
            status = memory_item.get("status", "")

            # 1. LLM이 뽑은 핵심 키워드가 현재 입력에 다시 등장하면 강하게 연결
            for keyword in keywords:
                keyword = str(keyword).strip()
                if keyword and keyword in user_input:
                    score += 3

            # 2. summary 안의 의미 있는 토큰이 현재 입력에 포함되면 약하게 연결
            for token in summary.split():
                token = token.strip(".,!?~…은는이가을를에의도만고")
                if len(token) >= 2 and token in user_input:
                    score += 1

            # 3. 이어지는 표현이면 active 이슈와 연결 가능성을 높임
            if any(word in user_input for word in ["아직도", "계속", "오늘도", "또", "그때처럼", "여전히"]):
                if status == "active":
                    score += 2

            # 4. category 자체가 입력에 직접 들어오는 경우는 드물지만 보조 점수로 둠
            if category and category != "none" and category in user_input:
                score += 1

            if score > 0:
                related.append((score, memory_item))

        related = sorted(related, key=lambda x: x[0], reverse=True)
        return [item for score, item in related[:max_results]]

    def format_for_prompt(self, related_memories):
        if not related_memories:
            return "이전 건강 이슈 메모리 없음"

        lines = []
        for i, memory_item in enumerate(related_memories, start=1):
            detail = memory_item.get("detail", {}) or {}
            keywords = detail.get("keywords", []) or []
            lines.append(
                f"{i}. 날짜: {memory_item['created_at']} / "
                f"분류: {memory_item['category']} / "
                f"요약: {memory_item['summary']} / "
                f"키워드: {', '.join(map(str, keywords))} / "
                f"상태: {memory_item['status']}"
            )
        return "\n".join(lines)

    def to_dataframe(self):
        return pd.DataFrame(self.memories)

print("멀티 사용자 건강 이슈 메모리 클래스 준비 완료")

멀티 사용자 건강 이슈 메모리 클래스 준비 완료


## 대화 내역 DB에 저장하기

In [ ]:
# conversations 테이블 생성

import sqlite3
from datetime import datetime

def create_conversations_table(db_path='dandibom.db'):
    """conversations 테이블을 생성합니다."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS conversations (
            conv_id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,
            role VARCHAR(10) NOT NULL CHECK(role IN ('user', 'assistant')),
            message TEXT NOT NULL,
            created_at TEXT NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(user_id)
        )
    ''')
    conn.commit()
    conn.close()
    print("Conversations table ensured to exist.")

create_conversations_table()

In [ ]:
def save_conversation_to_db(user_id, role, message, db_path='dandibom.db'):
    """대화 내역을 conversations 테이블에 저장합니다."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            "INSERT INTO conversations (user_id, role, message, created_at) VALUES (?, ?, ?, ?)",
            (user_id, role, message, created_at)
        )
        conn.commit()
        if role == 'user':
            print(f"User message for user_id {user_id} saved to DB.")
        else:
            print(f"Assistant message for user_id {user_id} saved to DB.")
    except Exception as e:
        print(f"Error saving conversation to DB: {e}")
    finally:
        conn.close()

# 경상도 사투리 테스트

아래 테스트는 순서대로 실행하면 된다.  
특히 첫 번째에서 무릎 이슈를 저장한 뒤, 뒤쪽 대화에서 "아직도", "오늘도" 같은 표현이 이전 건강 이슈와 연결되는지 확인한다.


In [ ]:
# DB 불러오기
conn = sqlite3.connect('dandibom.db')
cursor = conn.cursor()

In [ ]:
# 에러 방지를 위해 일단 테이블 삭제했다 재생성 필요

cursor.execute('DROP TABLE IF EXISTS users;')
cursor.execute('DROP TABLE IF EXISTS conditions;')

cursor.execute('''
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    role TEXT,
    name TEXT,
    code TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS conditions (
    cond_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    meal TEXT,
    sleep TEXT,
    mood TEXT,
    pain TEXT,
    medicine TEXT,
    recorded_at DATE
)
''')

## 테스트 1

In [ ]:
# 사용자 데이터 추가
users_data = [
    (1, '대상자', '김복순', 'user'),
    (2, '대상자', '박영감', 'user')
]
cursor.executemany("INSERT OR IGNORE INTO users VALUES (?,?,?,?)", users_data)

# 건강 상태 데이터 추가 (새로운 스키마에 맞게)
conditions_data = [
    (1, '잘 먹음', '푹 잠', '좋음', '무릎 약간 불편', '고혈압약 복용 중', '2024-07-25'),
    (1, '보통', '설침', '괜찮음', '없음', '고혈압약 복용 중', '2024-07-26'),
    (2, '소식', '푹 잠', '평온', '없음', '당뇨약 복용 중', '2024-07-25'),
    (2, '보통', '푹 잠', '평온', '없음', '당뇨약 복용 중', '2024-07-26')
]
cursor.executemany("INSERT OR IGNORE INTO conditions (user_id, meal, sleep, mood, pain, medicine, recorded_at) VALUES (?,?,?,?,?,?,?)", conditions_data)

conn.commit()
conn.close()

# 데이터 다시 로드하여 최신 상태 반영
users_df, conditions_df = load_user_data_from_db()

print("Updated users data:")
display(users_df)
print("Updated conditions data:")
display(conditions_df)

test_queries_user1 = [
    "어제부터 무릎이 아직도 시큰그리하다.",
    "점심은 아까 국에 밥 말아 뭇다.",
    "오늘도 좀 불편하네.",
]

test_queries_user2 = [
    "아침에 혈당 수치가 높게 나와서 걱정이야.",
    "요 며칠 밤에 잠을 설쳐서 피곤하네."
]

print("\n========== 김복순 어르신 (user_id=1)과의 대화 시작 ==========")
for i, query in enumerate(test_queries_user1, start=1):
    print(f"\n----- 김복순 어르신 테스트 {i} -----")
    chat_with_llm(user_id=1, user_query=query)

print("\n========== 박영감 어르신 (user_id=2)과의 대화 시작 ==========")
for i, query in enumerate(test_queries_user2, start=1):
    print(f"\n----- 박영감 어르신 테스트 {i} -----")
    chat_with_llm(user_id=2, user_query=query)


print("\n========== user_id=1 저장된 건강 이슈 메모리 ==========")
if 1 in health_memories_by_user:
    display(health_memories_by_user[1].to_dataframe())
else:
    print("user_id=1 건강 이슈 메모리 없음")

print("\n========== user_id=2 저장된 건강 이슈 메모리 ==========")
if 2 in health_memories_by_user:
    display(health_memories_by_user[2].to_dataframe())
else:
    print("user_id=2 건강 이슈 메모리 없음")

print("\n========== user_id=1 최근 대화 메모리 ==========")
if 1 in chat_histories_by_user:
    display(chat_histories_by_user[1].messages)
else:
    print("user_id=1 최근 대화 메모리 없음")

print("\n========== user_id=2 최근 대화 메모리 ==========")
if 2 in chat_histories_by_user:
    display(chat_histories_by_user[2].messages)
else:
    print("user_id=2 최근 대화 메모리 없음")

Updated users data:


,user_id,role,name,code
0,1,대상자,김복순,user
1,2,대상자,박영감,user


Updated conditions data:


,cond_id,user_id,meal,sleep,mood,pain,medicine,recorded_at
0,1,1,잘 먹음,푹 잠,좋음,무릎 약간 불편,고혈압약 복용 중,2024-07-25
1,2,1,보통,설침,괜찮음,없음,고혈압약 복용 중,2024-07-26
2,3,2,소식,푹 잠,평온,없음,당뇨약 복용 중,2024-07-25
3,4,2,보통,푹 잠,평온,없음,당뇨약 복용 중,2024-07-26



========== 김복순 어르신 (user_id=1)과의 대화 시작 ==========

----- 김복순 어르신 테스트 1 -----

[사용자 ID]
1

[사용자 입력]
어제부터 무릎이 아직도 시큰그리하다.

[사용자 이름: 김복순, 기존 건강 상태: 최근 기록: 식사- 보통, 수면- 설침, 기분- 괜찮음, 통증- 없음, 복약- 고혈압약 복용 중.]

[LLM 답변]
어르신, 무릎이 시큰거리시다니 걱정입니다. 언제부터 그러셨나요? 무리하지 않도록 조심하시고, 계속 불편하시면 주변에 말씀해 보시는 것도 좋으실 것 같습니다.

----- 김복순 어르신 테스트 2 -----

[사용자 ID]
1

[사용자 입력]
점심은 아까 국에 밥 말아 뭇다.

[사용자 이름: 김복순, 기존 건강 상태: 최근 기록: 식사- 보통, 수면- 설침, 기분- 괜찮음, 통증- 없음, 복약- 고혈압약 복용 중.]

[LLM 답변]
어르신, 점심으로 국에 밥 말아 드셨다니 잘하셨습니다. 여름철에는 시원한 국물이 속을 편안하게 해주지요. 무릎은 조금 괜찮아지셨나요? 계속 불편하시면 꼭 주변에 말씀해 보시는 것이 좋습니다.

----- 김복순 어르신 테스트 3 -----

[사용자 ID]
1

[사용자 입력]
오늘도 좀 불편하네.

[사용자 이름: 김복순, 기존 건강 상태: 최근 기록: 식사- 보통, 수면- 설침, 기분- 괜찮음, 통증- 없음, 복약- 고혈압약 복용 중.]

[LLM 답변]
어르신, 오늘도 불편하시다니 걱정입니다. 무릎 통증이 계속 되시는 건가요? 무리하지 않도록 잘 챙기시고, 조금이라도 더 불편하시다면 주변에 꼭 말씀해 보시는 것이 좋습니다.

========== 박영감 어르신 (user_id=2)과의 대화 시작 ==========

----- 박영감 어르신 테스트 1 -----

[사용자 ID]
2

[사용자 입력]
아침에 혈당 수치가 높게 나와서 걱정이야.

[사용자 이름: 박영감, 기존 건강 상태: 최근 기록: 식사- 보통, 수면- 푹 잠, 기분- 평온, 통증- 

...

,created_at,category,summary,detail,status,original_input,assistant_answer
0,2026-07-14 07:36,health,아침에 혈당 수치가 높게 나왔다는 이슈.,"{'keywords': ['혈당', '걱정'], 'time_expr': '아침', ...",active,아침에 혈당 수치가 높게 나와서 걱정이야.,아침에 혈당 수치가 높게 나왔다니 걱정입니다. 요즘 건강은 어떠신가요? 혹시 혈당 ...
1,2026-07-14 07:36,sleep,어르신이 최근 몇 일간 잠을 설쳐서 피곤하다고 하셨습니다.,"{'keywords': ['수면', '피곤'], 'time_expr': '며칠 전부...",active,요 며칠 밤에 잠을 설쳐서 피곤하네.,"어르신, 요 며칠 밤에 잠을 설쳐서 피곤하시다니 걱정입니다. 혹시 어떤 이유로 잠을..."



========== user_id=1 최근 대화 메모리 ==========


[HumanMessage(content='어제부터 무릎이 아직도 시큰그리하다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='어르신, 무릎이 시큰거리시다니 걱정입니다. 언제부터 그러셨나요? 무리하지 않도록 조심하시고, 계속 불편하시면 주변에 말씀해 보시는 것도 좋으실 것 같습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='점심은 아까 국에 밥 말아 뭇다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='어르신, 점심으로 국에 밥 말아 드셨다니 잘하셨습니다. 여름철에는 시원한 국물이 속을 편안하게 해주지요. 무릎은 조금 괜찮아지셨나요? 계속 불편하시면 꼭 주변에 말씀해 보시는 것이 좋습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='오늘도 좀 불편하네.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='어르신, 오늘도 불편하시다니 걱정입니다. 무릎 통증이 계속 되시는 건가요? 무리하지 않도록 잘 챙기시고, 조금이라도 더 불편하시다면 주변에 꼭 말씀해 보시는 것이 좋습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


========== user_id=2 최근 대화 메모리 ==========


[HumanMessage(content='아침에 혈당 수치가 높게 나와서 걱정이야.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='아침에 혈당 수치가 높게 나왔다니 걱정입니다. 요즘 건강은 어떠신가요? 혹시 혈당 수치가 높아진 것에 대해 어떤 조치를 생각하고 계신가요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='요 며칠 밤에 잠을 설쳐서 피곤하네.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='어르신, 요 며칠 밤에 잠을 설쳐서 피곤하시다니 걱정입니다. 혹시 어떤 이유로 잠을 잘 못 주무셨나요? 무리하지 마시고, 편안한 휴식 취하시길 바랍니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 테스트 2

In [ ]:
# 새로운 사용자 추가 (user_id=3)
conn = sqlite3.connect('dandibom.db')
cursor = conn.cursor()

users_data_test2 = [
    (3, '대상자', '최필자', 'user')
]
# INSERT OR IGNORE를 사용하여 이미 존재하면 건너뜁니다.
cursor.executemany("INSERT OR IGNORE INTO users VALUES (?,?,?,?)", users_data_test2)
conn.commit()
conn.close()

# 사용자 데이터를 다시 로드하여 최신 상태 반영
# chat_with_llm 함수가 users_df와 conditions_df를 참조하기 때문입니다.
users_df, conditions_df = load_user_data_from_db()
print("Updated users data for Test 2 (including user_id=3):")
display(users_df)

# user_id=3을 사용하여 샘플 대화 진행
print("\n========== 최필자 (user_id=3)과의 대화 시작 ==========")
sample_query_user3 = "오늘 아침에 허리가 좀 삐끗한 것 같아. 괜찮을랑가."
llm_response_user3 = chat_with_llm(user_id=3, user_query=sample_query_user3)

print("\n========== user_id=3 대화 내역 확인 ==========")
conn = sqlite3.connect('dandibom.db')
conversations_df = pd.read_sql_query(f"SELECT * FROM conversations WHERE user_id = 3", conn)
conn.close()

if not conversations_df.empty:
    print("conversations 테이블에 저장된 user_id=3의 대화 내역:")
    display(conversations_df)
else:
    print(f"user_id=3의 대화 내역이 conversations 테이블에 저장되지 않았습니다.")

Updated users data for Test 2 (including user_id=3):


,user_id,role,name,code
0,3,대상자,최필자,user



========== 최필자 (user_id=3)과의 대화 시작 ==========
User message for user_id 3 saved to DB.
Assistant message for user_id 3 saved to DB.

[사용자 ID]
3

[사용자 입력]
오늘 아침에 허리가 좀 삐끗한 것 같아. 괜찮을랑가.

[사용자 이름: 최필자, 기존 건강 상태: 특별한 건강 이슈 없음]

[LLM 답변]
어르신, 허리가 삐끗하셨다니 걱정입니다. 무리하지 않도록 조심하시고, 가능하면 편안한 자세로 쉬시는 게 좋으실 것 같습니다. 괜찮아지시길 바랍니다.

========== user_id=3 대화 내역 확인 ==========
conversations 테이블에 저장된 user_id=3의 대화 내역:


,conv_id,user_id,role,message,created_at
0,1,3,user,오늘 아침에 허리가 좀 삐끗한 것 같아. 괜찮을랑가.,2026-07-15 06:24:03
1,2,3,assistant,"어르신, 허리가 삐끗하셨다니 걱정입니다. 무리하지 않도록 조심하시고, 가능하면 편안...",2026-07-15 06:24:03
